# 03：一个损失函数，两个学习系统

**适合谁：**了解基本 PyTorch 的开发者。如果刚接触梯度，请先学习示例 02。

**你的任务：**把经典编码器与量子层连接起来，证明同一个优化器会更新两者。
实验将得到留出测试数据上的预测、梯度图以及简洁的实验报告。
这个玩具问题用于展示联合训练，不代表量子优势。

**运行前想一想：**量子计算位于模型中间时，经典编码器要收到梯度，需要具备什么条件？

[English edition](../notebooks/03_hybrid_model.ipynb)

**学习目标：** 将一批数据送入经典层与量子层，并验证两层通过同一个损失共同学习。

**先修要求：** 已完成 02，并了解 PyTorch Linear、Sequential 和批维度。这是本地 CPU 实验，不是真机训练循环。

学习路线见 [workshop 导航](../README.md)。


## 1. 构造训练和测试输入

目标函数是 cos(x)，因此可以独立于实现计算正确答案。
将相邻采样点交替分入训练集和测试集。测试点仍在训练范围内，所以这里检验的是插值，而不是外推。


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import torch
import flagquantum as fq

all_x = torch.linspace(-2, 2, 41)
x_train, x_test = all_x[::2], all_x[1::2]
y_train, y_test = torch.cos(x_train), torch.cos(x_test)
plt.scatter(x_train, y_train, label="Train", marker="o")
plt.scatter(x_test, y_test, label="Test", marker="x")
plt.xlabel("Input x")
plt.ylabel("Target cos(x)")
plt.legend()
plt.show()


## 2. 编写量子程序

经典编码器为每个输入生成一个角度。我们再加上可训练的量子偏移量，然后施加 RY 门。
电路的 Z 期望值就是模型预测。`bsz` 为输入批次中的每一项创建一个电路状态。


In [ ]:
def quantum_program(parameters, encoded):
    circuit = fq.Circuit(1, bsz=len(encoded))
    circuit.ry(0, encoded[:, 0] + parameters["offset"][0])
    return circuit


quantum = fq.Module(
    quantum_program,
    parameters={"offset": (1,)},
    init="uniform",
    seed=42,
    policy=fq.RuntimePolicy(
        execution_options=fq.ExecutionOptions(mode="statevector", device="cpu"),
        observable="z_sum",
        observable_qubits=(0,),
    ),
)


## 3. 连接普通 PyTorch 层

编码器将 x 映射为 wx+b，量子层再将这个角度映射为测量期望值。
因此，这个具体模型计算的是 cos(wx+b+offset)。梯度可以连续地穿过两层。

**先预测：**仅凭这些数据，能唯一确定 b 和 offset 吗？观察训练后的参数时，请记住这个问题。

电路图中，每条横线对应一个量子比特，量子门从左到右执行。图中的上下顺序不等于云平台结果位串的排列顺序。 这里只画初始参数下的电路，不在训练循环中重复绘图。 图中展示训练集第一个输入经初始经典编码器生成的角度；其他输入的角度可能不同。

**概念：** 输入一批数据，就得到一批预测。例如 x=[-1, 0, 1] 的形状是 (3,)；`x[:, None]` 将其变成经典 Linear 层要求的 (3, 1)。输出 `encoded` 为 (3, 1)，`encoded[:, 0]` 取出三个角度。`bsz=3` 表示三个独立电路状态，而不是三个量子比特。

**预期输出：** 经典编码器生成角度，量子层为每个输入返回一个 Z 期望值。`reshape(-1)` 将三个预测整理成 (3,) 以匹配三个目标值，不改变数值。

**常见误解：** 批大小与量子比特数量是两回事。两层通过同一条本地梯度路径训练，输出是期望值，而不是概率。


In [ ]:
encoder = torch.nn.Linear(1, 1)
with torch.no_grad():
    encoder.weight.fill_(0.7)
    encoder.bias.fill_(0.2)
model = torch.nn.Sequential(encoder, quantum)
initial_parameters = {name: p.detach().clone() for name, p in model.named_parameters()}
with torch.no_grad():
    encoded_preview = encoder(x_train[:1, None])
    initial_circuit = quantum_program(quantum.parameter_groups, encoded_preview)
fig, ax = initial_circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
initial_prediction = model(x_test[:, None]).detach().reshape(-1)
print("Prediction shape:", initial_prediction.shape)
print("Trainable parameters:")
for name, p in model.named_parameters():
    print(name, p.detach())


## 4. 训练前先检查一次反向传播

损失下降并不能单独证明两个层都在学习。先直接查看各层的梯度。
以 `0.` 开头的参数属于编码器，以 `1.` 开头的参数属于量子层。


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
optimizer.zero_grad()
prediction = model(x_train[:, None]).reshape(-1)
loss = (prediction - y_train).square().mean()
loss.backward()
for name, p in model.named_parameters():
    assert p.grad is not None and torch.isfinite(p.grad).all()
    print(name, "gradient:", p.grad)
assert encoder.weight.grad.abs().sum().item() > 0
assert sum(p.grad.abs().sum().item() for p in quantum.parameters()) > 0


## 5. 用一个优化器训练

同时记录训练损失、测试损失和各层的梯度范数。测试误差只用于评估，不用于更新参数。
要进行全新实验，请从模型构造开始重新运行；只重复运行此单元会继续训练现有模型。

`zero_grad()` 让每一步从空梯度开始。`torch.no_grad()` 只在代码块内关闭梯度记录，使测试集评估不再构建训练图；它不会更新参数，也不会阻止之后继续训练。`detach()` 将记录用的张量与训练图分离；训练预测仍须保留梯度连接。


In [ ]:
train_losses, test_losses = [], []
encoder_gradients, quantum_gradients = [], []
for step in range(150):
    optimizer.zero_grad()
    prediction = model(x_train[:, None]).reshape(-1)
    loss = (prediction - y_train).square().mean()
    loss.backward()
    encoder_gradients.append(
        sum(p.grad.square().sum().item() for p in encoder.parameters()) ** 0.5
    )
    quantum_gradients.append(
        sum(p.grad.square().sum().item() for p in quantum.parameters()) ** 0.5
    )
    train_losses.append(loss.detach().item())
    optimizer.step()
    with torch.no_grad():
        test_prediction = model(x_test[:, None]).reshape(-1)
        test_losses.append((test_prediction - y_test).square().mean().item())

final_prediction = model(x_test[:, None]).detach().reshape(-1)
parameter_changes = {
    name: (p.detach() - initial_parameters[name]).norm().item()
    for name, p in model.named_parameters()
}
print("Test MSE:", test_losses[-1])
print("Parameter changes:", parameter_changes)
assert test_losses[-1] < 0.01
assert any(
    delta > 1e-5 for name, delta in parameter_changes.items() if name.startswith("0.")
)
assert any(
    delta > 1e-5 for name, delta in parameter_changes.items() if name.startswith("1.")
)


## 6. 读取三类证据

第一幅图检查留出测试输入上的预测，第二幅图展示优化过程，
第三幅图展示梯度确实到达了模型的两个部分。拟合良好固然有用，但任何一幅图都不能单独证明这三件事。

**读懂输出：** `Prediction shape: torch.Size([20])` 表示 20 个测试输入各有一个预测，不是 20 个量子比特。默认设置下，`Test MSE` 应小于 0.01，两组参数都应出现非零变化。较低测试误差说明在这个范围内拟合有效，不代表量子优势。

**安全重复运行：** 重建编码器、量子层和优化器才是新的训练；仅重跑训练循环会继续训练已有模型。


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
ax[0].plot(x_test, y_test, label="Target")
ax[0].scatter(x_test, initial_prediction, label="Before", s=15)
ax[0].scatter(x_test, final_prediction, label="After", s=15)
ax[0].set(xlabel="Held-out x", ylabel="Prediction")
ax[0].legend()
ax[1].semilogy(train_losses, label="Train (before update)")
ax[1].semilogy(test_losses, label="Test (after update)")
ax[1].set(xlabel="Step", ylabel="MSE")
ax[1].legend()
ax[2].plot(encoder_gradients, label="Classical encoder")
ax[2].plot(quantum_gradients, label="Quantum layer")
ax[2].set(xlabel="Step", ylabel="Gradient norm")
ax[2].legend()
plt.tight_layout()
plt.show()


## 7. 保留一个能解释清楚的结果

记录配置和证据，不要只保存图像。源码路径可以帮助发现误用其他安装版本的情况；公开报告前，请检查是否包含私人路径。


In [ ]:
from pathlib import Path

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next(
    (root / "flagos2026" for root in search_roots if (root / "flagos2026/notebooks").is_dir()),
    None,
)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks above the current working directory. Open this notebook from the deployed workshop tree.")
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import json

report = {
    "experiment": "hybrid_cosine_fit",
    "source": "local_simulation",
    "flagquantum_version": fq.__version__,
    "torch_version": torch.__version__,
    "quantum_seed": 42,
    "steps": 150,
    "learning_rate": 0.04,
    "train_points": len(x_train),
    "test_points": len(x_test),
    "test_mse": test_losses[-1],
    "parameter_changes": parameter_changes,
    "model": "cos(w*x+b+offset)",
}
(OUTPUTS / "hybrid_model_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))


## 向别人解释这个实验

- 梯度在哪里从量子层传回经典编码器？
- 为什么不同的 b 和 offset 会得到相同预测？这个模型只依赖它们的和，因此无法分别唯一确定这两个参数。
- 要声称这个模型优于经典方法，还需要什么证据？

**接下来尝试：**冻结一个部分，从相同初始值重新训练并比较结果。也可以测试 [-2,2] 之外的输入，但应明确称为外推实验。
继续学习[示例 04](04_one_problem_three_simulators.ipynb)，在保持目标不变的情况下切换模拟表示。
使用[实验记录](../FIELD_NOTES.md)写下结论。


## 带回家使用

这是完整的混合训练示例。经典编码器和量子参数属于同一个 PyTorch 模型，
也由同一个优化器更新。

```python
import torch
import flagquantum as fq

def quantum_program(parameters, inputs):
    return fq.Circuit(1, bsz=len(inputs)).ry(
        0, inputs[:, 0] + parameters["offset"][0],
    )

quantum = fq.Module(
    quantum_program, parameters={"offset": (1,)}, init="uniform", seed=42,
    policy=fq.RuntimePolicy(observable="z_sum", observable_qubits=(0,)),
)
encoder = torch.nn.Linear(1, 1)
with torch.no_grad():
    encoder.weight.fill_(0.7)
    encoder.bias.fill_(0.2)
model = torch.nn.Sequential(encoder, quantum)
x = torch.linspace(-2, 2, 21)[:, None]
target = torch.cos(x[:, 0])
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
for step in range(150):
    optimizer.zero_grad()
    loss = (model(x).reshape(-1) - target).square().mean()
    loss.backward()
    optimizer.step()
print("Training loss:", loss.detach().item())
```

上面的完整实验还包含留出评估，以及两个部分都发生参数更新的检查。
成功拟合余弦函数，不是一般量子优势的证据。
